# 4.1 怎麼知道字的順序？


「貓追狗」與「狗追貓」用了同樣三個字，角色卻交換了。[2.1](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html) 的字向量只由ID查出，相同字在不同位置會得到相同起始數字；而 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的因果許可決定誰能讀誰，也不是直接替每格提供「這是第幾個位置」的特徵。因此我們再準備一張位置表，讓文字內容與位置一起成為後續計算的線索。

每個文字位置也查出一個向量，與字向量逐格相加。若字向量有D數，位置向量也必須有D數，才能相加而不丟掉特徵。這叫位置表示（positional embedding）。本節用可調查表的絕對位置：位置0、1、2各有一列參數，訓練時可按答案代價改變，不是直接把位置編號乘進字ID。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

e = nn.Embedding(4, 3)
p = nn.Embedding(5, 3)
with torch.no_grad():
    e.weight.copy_(torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]))
    p.weight.copy_(torch.tensor([[0.0, 0.0, 0.0], [0.1, 0.2, 0.3], [0.2, 0.4, 0.6], [0.3, 0.6, 0.9], [0.4, 0.8, 1.2]]))
ids = torch.tensor([1, 2, 1])
positions = torch.arange(3)
x = e(ids) + p(positions)
print(positions)
print(x)
assert not torch.equal(x[0], x[2])

輸入ID為1、2、1，兩次ID1原先都查到 `[1,0,0]`。`torch.arange(3)` 建立位置編號 `[0,1,2]`，每個位置使用相應的p列。位置0加入 `[0,0,0]`，結果仍 `[1,0,0]`；位置1為 `[0.1,1.2,0.3]`；位置2的ID1則變 `[1.2,0.4,0.6]`。最後檢查兩個相同字的表示確實不再完全一樣。

這些手動小數只是讓例子透明，並不是某種公認的位置公式。正式模型中p通常從亂數開始，與字表一同學習。相加後的向量同時含有兩種資訊。後面的[Linear（線性層）](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html)會用權重把多個輸入特徵加權混合成新數字；它與注意力還需要在訓練中學會使用內容與位置線索。相加本身沒有確保模型已懂語序，也無法從一個數值看出它「知道第一個字」。

查表法還有明確的長度限制。`nn.Embedding(5,3)` 只提供位置0到4；若輸入位置5，就索引超出範圍。模型不能把沒學過的新位置悄悄當成舊位置重用，否則遠處的順序會被混淆。後續的其他位置方法會改變計算方式，眼前先用表格建立可檢驗的起點。

練習把所有位置向量清零。請在`with torch.no_grad():`區塊內，第二個`p.weight.copy_(...)`完整呼叫結束後新增`p.weight.zero_()`，行首保留與`copy_`相同的四個空格；下一行`ids = torch.tensor([1, 2, 1])`仍從最左邊開始。這個區塊包住手動設定參數的操作，背景可看[1.12](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)。同時把最後檢查改成`assert torch.equal(x[0], x[2])`。先預測相同ID1又會得到相同`[1,0,0]`，再從頭執行核對。這次字表沒改，變化完全來自位置資訊，能明確指出兩張表各貢獻了什麼。
